# Очистка `questions.csv`

**грейн**: одна строка = один вопрос (question_id)

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

In [2]:
con = duckdb.connect()

#RAW_DIR = Path("data/raw")
RAW_DIR = Path("RiiidAnswerCorrectnessPrediction")
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def sql_path(path: Path) -> str:
    return str(path.resolve()).replace("\\", "/").replace("'", "''")

def checkpoint(name, ok, detail=""):
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f" — {detail}"))

## 1. Пути

In [3]:
QUESTIONS_PATH = RAW_DIR / "questions.csv"
QUESTIONS_CLEAN_PATH = OUT_DIR / "questions_clean.parquet"
QUESTIONS_REJECTED_PATH = OUT_DIR / "questions_rejected.parquet"
QUESTIONS_DUPLICATES_PATH = OUT_DIR / "questions_duplicates.parquet"

assert QUESTIONS_PATH.exists(), f"Не найден файл: {QUESTIONS_PATH.resolve()}"

QUESTIONS = sql_path(QUESTIONS_PATH)
QUESTIONS_CLEAN = sql_path(QUESTIONS_CLEAN_PATH)
QUESTIONS_REJECTED = sql_path(QUESTIONS_REJECTED_PATH)
QUESTIONS_DUPLICATES = sql_path(QUESTIONS_DUPLICATES_PATH)

## 2. STG (сырье)

In [4]:
con.execute(f"""
CREATE OR REPLACE VIEW stg_questions AS
SELECT *
FROM read_csv(
    '{QUESTIONS}',
    header = true,
    all_varchar = true,
    nullstr = ''
)
""")

display(con.sql("SELECT * FROM stg_questions LIMIT 10").df())

,question_id,bundle_id,correct_answer,part,tags
0,0,0,0,1,51 131 162 38
1,1,1,1,1,131 36 81
2,2,2,0,1,131 101 162 92
3,3,3,0,1,131 149 162 29
4,4,4,3,1,131 5 162 38
5,5,5,2,1,131 149 162 81
6,6,6,2,1,10 94 162 92
7,7,7,0,1,61 110 162 29
8,8,8,3,1,131 13 162 92
9,9,9,3,1,10 164 81


## 3. ODS (типизация и нормализация)

In [5]:
con.execute("""
CREATE OR REPLACE TEMP VIEW typed_questions AS
SELECT
    TRY_CAST(question_id AS INTEGER) AS question_id,
    TRY_CAST(bundle_id AS INTEGER) AS bundle_id,
    TRY_CAST(correct_answer AS SMALLINT) AS correct_answer,
    TRY_CAST(part AS SMALLINT) AS part,
    NULLIF(
        regexp_replace(trim(tags), '\\s+', ' ', 'g'),
        ''
    ) AS tags
FROM stg_questions
""")

display(con.sql("SELECT * FROM typed_questions LIMIT 10").df())

,question_id,bundle_id,correct_answer,part,tags
0,0,0,0,1,51 131 162 38
1,1,1,1,1,131 36 81
2,2,2,0,1,131 101 162 92
3,3,3,0,1,131 149 162 29
4,4,4,3,1,131 5 162 38
5,5,5,2,1,131 149 162 81
6,6,6,2,1,10 94 162 92
7,7,7,0,1,61 110 162 29
8,8,8,3,1,131 13 162 92
9,9,9,3,1,10 164 81


## 4. Data Quality

- целостность обязательных полей;
- уникальность questin_id;
- диапазон должен быть:
  - для correct_answer ∈ {0,1,2,3};
  - для part ∈ [1,7];
- если есть теги, то они целые id через пробел (пустые данные допустимы).

In [6]:
dq = con.sql("""
SELECT
    COUNT(*) AS rows_total,
    COUNT(DISTINCT question_id) AS unique_question_ids,

    COUNT(*) FILTER (
        WHERE question_id IS NULL
           OR bundle_id IS NULL
           OR correct_answer IS NULL
           OR part IS NULL
    ) AS missing_mandatory,

    COUNT(*) FILTER (
        WHERE correct_answer NOT BETWEEN 0 AND 3
    ) AS bad_correct_answer,

    COUNT(*) FILTER (
        WHERE part NOT BETWEEN 1 AND 7
    ) AS bad_part,

    COUNT(*) FILTER (
        WHERE tags IS NOT NULL
          AND NOT regexp_full_match(tags, '[0-9]+( [0-9]+)*')
    ) AS bad_tags_format,

    COUNT(*) FILTER (WHERE tags IS NULL) AS null_tags
FROM typed_questions
""").df()

display(dq)

r = dq.iloc[0]
dup_rows = int(r["rows_total"] - r["unique_question_ids"])

checkpoint("Completeness", int(r["missing_mandatory"]) == 0,
           f"{int(r['missing_mandatory']):,} строк")
checkpoint("Uniqueness question_id", dup_rows == 0,
           f"{dup_rows:,} лишних строк")
checkpoint("correct_answer ∈ [0,3]", int(r["bad_correct_answer"]) == 0,
           f"{int(r['bad_correct_answer']):,} строк")
checkpoint("part ∈ [1,7]", int(r["bad_part"]) == 0,
           f"{int(r['bad_part']):,} строк")
checkpoint("Формат tags", int(r["bad_tags_format"]) == 0,
           f"{int(r['bad_tags_format']):,} строк")

,rows_total,unique_question_ids,missing_mandatory,bad_correct_answer,bad_part,bad_tags_format,null_tags
0,13523,13523,0,0,0,0,1


✅ Completeness
✅ Uniqueness question_id
✅ correct_answer ∈ [0,3]
✅ part ∈ [1,7]
✅ Формат tags


## 5. Отделяем невалидные строки

In [7]:
invalid_predicate = """
       question_id IS NULL
    OR bundle_id IS NULL
    OR correct_answer IS NULL
    OR part IS NULL
    OR correct_answer NOT BETWEEN 0 AND 3
    OR part NOT BETWEEN 1 AND 7
    OR (tags IS NOT NULL
        AND NOT regexp_full_match(tags, '[0-9]+( [0-9]+)*'))
"""

con.execute(f"""
COPY (
    SELECT *
    FROM typed_questions
    WHERE {invalid_predicate}
)
TO '{QUESTIONS_REJECTED}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Rejected:",
      con.sql(f"SELECT COUNT(*) FROM read_parquet('{QUESTIONS_REJECTED}')").fetchone()[0])

Rejected: 0


## 6. Дедупликация и сохранение clean-слоя

In [8]:
con.execute(f"""
COPY (
    SELECT * EXCLUDE (rn)
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY question_id
                ORDER BY bundle_id, correct_answer, part, tags NULLS LAST
            ) AS rn
        FROM typed_questions
        WHERE NOT ({invalid_predicate})
    )
    WHERE rn > 1
)
TO '{QUESTIONS_DUPLICATES}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.execute(f"""
COPY (
    SELECT
        question_id,
        bundle_id,
        correct_answer,
        part,
        tags
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY question_id
                ORDER BY bundle_id, correct_answer, part, tags NULLS LAST
            ) AS rn
        FROM typed_questions
        WHERE NOT ({invalid_predicate})
    )
    WHERE rn = 1
)
TO '{QUESTIONS_CLEAN}'
(FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Готово:", QUESTIONS_CLEAN_PATH.resolve())

Готово: /Users/user/Desktop/project/data/processed/questions_clean.parquet


## 7. Финальная проверка

In [9]:
final_dq = con.sql(f"""
SELECT
    COUNT(*) AS rows_total,
    COUNT(DISTINCT question_id) AS unique_question_ids,
    COUNT(*) FILTER (
        WHERE question_id IS NULL
           OR bundle_id IS NULL
           OR correct_answer NOT BETWEEN 0 AND 3
           OR part NOT BETWEEN 1 AND 7
    ) AS bad_rows
FROM read_parquet('{QUESTIONS_CLEAN}')
""").df()

display(final_dq)

r = final_dq.iloc[0]
checkpoint("Финал: question_id уникален",
           int(r["rows_total"]) == int(r["unique_question_ids"]))
checkpoint("Финал: обязательные домены валидны",
           int(r["bad_rows"]) == 0)

,rows_total,unique_question_ids,bad_rows
0,13523,13523,0


✅ Финал: question_id уникален
✅ Финал: обязательные домены валидны


## 8. Результат

- `data/processed/questions_clean.parquet` — очищенный справочник;
- `data/processed/questions_rejected.parquet` — технически невалидные строки;
- `data/processed/questions_duplicates.parquet` — лишние версии повторившихся `question_id`.